# Session Log — Accelerometer Entropy Adaptive PPG Sampling

**Project:** EMBC 2026 poster — accelerometer compressibility/entropy as a predictor of HR estimation error  
**Dataset:** IEEE Signal Processing Cup 2015 wrist PPG (22 recordings, 125 Hz)  
**Language:** Python 3.11 / Jupyter, conda env `ppg-entropy`  

---

## Session template (copy for each new session)

```
### Session YYYY-MM-DD
**Goal:** ...  
**Done:**
- 

**Key findings:**
- 

**Blockers / open questions:**
- 

**Next session:**
- 
```

---

### Session 2026-04-18
**Goal:** Project setup and Step 1 completion (data loading + sanity check)

**Done:**
- Confirmed dataset layout: `Training_data/` (12 T1 treadmill recordings) + `TestData/` (10 arm-movement recordings) + `TrueBPM/` (ground truth for test set)
- **Row-layout discrepancy caught and fixed:** training files have `sig` shape `(6, N)` with ECG at row 0, but test files have `sig` shape `(5, N)` with no ECG (rows 0–1 = PPG, 2–4 = ACC). The TestData Readme confirms ECG was withheld by the competition. Loader now branches on `sig.shape[0]` and asserts the shape matches the group.
- **Refined from two groups to three groups** after seeing raw ACC magnitudes:
  - T1 — treadmill (rec 01–12, training, 12 recs)
  - T2 — mixed arm exercise, `TEST_S*_T01.mat` (rec 13, 14, 19, 22 — shake/stretch/push-up/run/jump, 4 recs)
  - T3 — boxing, `TEST_S*_T02.mat` (rec 15, 16, 17, 18, 20, 21 — intensive forearm/upper-arm, 6 recs)
- Registered `Python (ppg-entropy)` as a Jupyter kernel and set it as the notebook default
- Added "Expected output" markdown before every code cell so the user can sanity-check each step independently
- Notebook runs end-to-end with 0 errors and 0 window/GT count mismatches

**Key findings:**
- Mean RMS ACC magnitude by group (raw, still contains 1 g gravity DC):
  - T1 treadmill: **1.476 g** (tight range 1.31–1.72)
  - T2 mixed arm: **1.068 g** (range 0.99–1.13)
  - T3 boxing:    **1.790 g** (range 1.47–2.02)
- Critical observation: **T3 magnitude ≥ T1 magnitude**, but T3 motion is irregular while T1 is periodic. This is exactly the discrimination problem entropy should solve — supports the paper thesis, not against it.
- Two-group analysis (treadmill vs T2+T3 pooled) gave means that were essentially tied (1.48 vs 1.50), which is why the three-group split tells a cleaner narrative.

**Blockers / open questions:**
- Gravity DC bias (~1 g) is still included in raw ACC magnitude. Deferred decision: whether to subtract it (high-pass or per-window mean removal) before computing RMS. Revisit after Step 4 entropy results to see if raw vs AC-only changes anything.
- Window-to-GT alignment is currently exact for all 22 recordings — no tricky off-by-one handling needed in Step 2.

**Next session:**
- Start Step 2: bandpass-filtered FFT HR estimator (stateless, per-window)
- Use the three-group color scheme (`steelblue` / `goldenrod` / `crimson`) in all MAE plots
- Report MAE per recording and per group (T1, T2, T3 separately)


---

## Research roadmap

| Step | Notebook | Status |
|------|----------|--------|
| 1 — Data loading & sanity check | `step1_data_loading.ipynb` | **done (2026-04-18)** — all 22 recordings loaded, 3-group split in place, row-layout fix applied, kernel registered |
| 2 — Simple HR estimator (FFT) | `step2_hr_estimator.ipynb` | not started |
| 3 — Exploratory plots (ACC mag vs error) | `step3_exploratory.ipynb` | not started |
| 4 — Entropy metrics | `step4_entropy.ipynb` | not started |
| 5 — Four-quadrant analysis | `step5_quadrant.ipynb` | not started |
| 6 — Summary comparison table | `step6_summary_table.ipynb` | not started |

**Three activity groups** (refined after seeing the data):

| Group | Label | Color | Recordings | Raw mean ACC RMS (g) |
|-------|-------|-------|------------|----------------------|
| T1 | treadmill | `steelblue` | 12 (IDs 1–12, `DATA_*_TYPE*`) | 1.476 |
| T2 | mixed arm exercise | `goldenrod` | 4 (IDs 13, 14, 19, 22, `TEST_S*_T01`) | 1.068 |
| T3 | boxing | `crimson` | 6 (IDs 15, 16, 17, 18, 20, 21, `TEST_S*_T02`) | 1.790 |

**Core hypothesis:** ACC compressibility/entropy is a better predictor of HR estimation error than ACC magnitude alone.

**Key comparison cases under the three-group split:**
- T1 treadmill: high magnitude, **low** entropy → entropy predicts low error (magnitude wrongly predicts high error)
- T2 mixed arm: low magnitude, moderate entropy → mixed
- T3 boxing: high magnitude, **high** entropy → both predict high error (they agree here)

The critical discrimination is **T1 vs T3**: they have similar magnitudes but opposite entropies, and they should have very different HR-estimation error.
